In [123]:
#import packages
import numpy as np
import scipy
import matplotlib.pyplot as plt
import pandas as pd
import xarray as xr
import os
import yaml
import itertools
import shutil
from datetime import datetime
from pathlib import Path
from openpyxl.styles import PatternFill
from openpyxl import load_workbook
from collections import defaultdict
import sys
from openpyxl.styles import Border, Side
from openpyxl.styles import Alignment, Font
from openpyxl.utils import get_column_letter
from numpy._core.numeric import indices
from tqdm import tqdm
from functools import lru_cache
from collections import deque
import networkx as nx
from collections import Counter


#set working dir
os.chdir("/Users/quinnmackay/Documents/GitHub/BICC/Antarctic Chronology Accuracy Project")

In [124]:
# load cores
project = 'Antarctic'
output_dir = 'table_out/'

# get all link combos
with open(f'/Users/quinnmackay/Documents/GitHub/BICC/Antarctic Chronology Accuracy Project/{project}/parameters.yml') as f:
    data = yaml.safe_load(f)
list_sites = data["list_sites"]
pairs = [f"{a}-{b}" for a, b in itertools.combinations(list_sites, 2)]

#define error margins
error_margin = 0.15

In [125]:
#Load in Big_Table for code indexing later.

original_big_table = pd.DataFrame()
all_links_count = {}
all_links_foragesort = {}
all_links_total = {}

for core in list_sites: # loop through each core
    for comparison_core in list_sites: # loop through each core other than the initial load
        pair = f"{core}-{comparison_core}"
        if core != comparison_core and pair in pairs: # make sure not the same core and we skip non-existent linkages
            pair_dir = Path(f'/Users/quinnmackay/Documents/GitHub/BICC/Antarctic Chronology Accuracy Project/{project}/{pair}')

            # Check: directory exists AND contains at least one .txt file
            txt_files = list(pair_dir.glob("*.txt"))
            if not pair_dir.is_dir() or not txt_files:
                continue

            dfs=[] #load all text files into one
            for txt in txt_files:
                df = pd.read_csv(txt, sep="\t", comment="#")
                dfs.append(df)
    
            num_files = len(dfs)
            load_data = pd.concat(dfs, ignore_index=True)
            original_rows = len(load_data)

            drop_rows = []
            drop_rows_merge = set()
            new_merged_rows = []
            for idx, row in load_data.iterrows():

                mask1 = abs(row['depth1'] - load_data['depth1']) <= error_margin
                mask1[idx] = False
                mask2 = abs(row['depth2'] - load_data['depth2']) <= error_margin 
                mask2[idx] = False

                close_points = load_data[mask1 & mask2]
                num_close = len(close_points)
                close_idxs = load_data.index[mask1 & mask2]

                if num_close > 0:
                    refs = [load_data.at[idx, 'reference']] + [load_data.at[i, 'reference'] for i in close_idxs] #adjoin references
                    merged_ref = "; ".join(str(r) for r in refs if pd.notna(r))

                    depth1_vals = [load_data.at[idx, 'depth1']] + [load_data.at[i, 'depth1'] for i in close_idxs]
                    merged_depth1 = np.round(np.mean(depth1_vals), 4)

                    depth2_vals = [load_data.at[idx, 'depth2']] + [load_data.at[i, 'depth2'] for i in close_idxs]
                    merged_depth2 = np.round(np.mean(depth2_vals), 4)

                    new_merged_rows.append({'reference': merged_ref, 'depth1': merged_depth1, 'depth2': merged_depth2}) #create new merged row

                    drop_rows_merge.add(idx)
                    for i in close_idxs:
                        drop_rows.append(i)

            # drop duplicate rows
            drop_rows = set(drop_rows).union(drop_rows_merge)
            load_data = load_data.drop(index=drop_rows).reset_index(drop=True)
            # add merged rows
            merged_df = pd.DataFrame(new_merged_rows)
            load_data = pd.concat([load_data, merged_df], ignore_index=True)
            load_data.drop_duplicates(subset=['depth1', 'depth2'], inplace=True)
            load_data = load_data.reset_index(drop=True)

            load_data = load_data.sort_values(by=['depth1']).reset_index(drop=True)
        
            #set up pair code stuff
            load_data[f"{pair}_code"] = [f"{pair}_{idx}" for idx in load_data.index]

            #save all the links for this pair
            all_links_total[f'{pair}'] = load_data[['depth1', 'depth2']].copy(deep=True)
            all_links_total[f'{pair}'] = all_links_total[f'{pair}'].rename(columns={
                'depth1': pair.split("-")[0],
                'depth2': pair.split("-")[1]})

            # rename to create unique columns for this pair
            load_data = load_data.rename(columns={
                'depth1': f"{pair}_{core}",
                'depth2': f"{pair}_{comparison_core}",
                'reference': f"{pair}_reference",
            })

            # append rows (block)
            original_big_table = pd.concat([original_big_table, load_data], axis=0, ignore_index=True)

#if core doesn't exist in all_links_count, add it with 0 val
for core in list_sites:
    if core not in all_links_count:
        all_links_count[core] = 0

print('Original Big Table loaded and processed.')

Original Big Table loaded and processed.


In [126]:
#useful function for getting the reference, index, and depths for a given code
def get_tie_info(code, depths=False):
    pair = code.split("_")[0] # Extract pair name from code
    row_vals = original_big_table.loc[original_big_table[f'{pair}_code'] == code].iloc[0] # Find row in original_big_table matching the code
    row_vals = row_vals.dropna() # Remove NaN values from the row
    
    reference = row_vals[f'{pair}_reference'] # Extract reference and row index
    index = row_vals.name

    print(row_vals)
    
    if depths: #get depths for the network
        depth1 = row_vals.iloc[0]
        depth2 = row_vals.iloc[1]
        return reference, index, depth1, depth2
    
    return reference, index

In [127]:
network_results = pd.read_excel(f'/Users/quinnmackay/Documents/GitHub/BICC/Antarctic Chronology Accuracy Project/table_out/{project}_full.xlsx', sheet_name=0, usecols=[0,21], index_col=0)
code_networks = pd.read_excel(f'/Users/quinnmackay/Documents/GitHub/BICC/Antarctic Chronology Accuracy Project/table_out/{project}_full.xlsx', sheet_name=1, usecols='S:AC', index_col=0)


#below code separates out the rows with errors for BFS processing
index_with_errors = (
    network_results['Within Row Errors'].notna()
    & network_results['Within Row Errors'].astype(str).str.strip().ne('')
)
network_results = network_results[index_with_errors]
error_indices = np.array(network_results.index.tolist())

In [128]:
#bfs functions

def get_depth(core, index): #function to grab the depth based on core and index
    row_vals = big_table.loc[index]
    non_zero_columns = row_vals[(row_vals.notna()) & (row_vals != 0)].index.tolist()

    core_columns = [col for col in non_zero_columns if col.endswith(core)]
    if not core_columns:
        print(f"No active columns found for core {core} in row {index}.")
        sys.exit()

    column_name = core_columns[0]
    return big_table.at[index, column_name]

def get_code(index): #function to grab the code based on index
    row_vals = big_table.loc[index]

    non_zero_columns = row_vals[(row_vals.notna()) & (row_vals != 0)].index.tolist()

    code_columns = [c for c in non_zero_columns if c.endswith("code")]
    if not code_columns:
        print(f"No code column found for row {index}.")
        sys.exit()

    return big_table.at[index, code_columns[0]]

def get_reference(index): #function to grab the reference based on index
    row_vals = big_table.loc[index]

    non_zero_columns = row_vals[(row_vals.notna()) & (row_vals != 0)].index.tolist()

    reference_columns = [c for c in non_zero_columns if c.endswith("reference")]
    if not reference_columns:
        print(f"No reference column found for row {index}.")
        sys.exit()

    return big_table.at[index, reference_columns[0]]

def get_core_pair(node): #takes a node (which is a tuple of (index, root_core)) and outputs the other pair in the same row of big_table
    idx, root_core = node

    root_core_row = big_table.loc[idx].dropna() #locates the row
    if root_core_row.index[0].split("_")[0].split("-")[0] == root_core: #if the first part of the pair is the root core, return the second part
        root_core_pair = root_core_row.index[0].split("_")[0].split("-")[1]
    elif root_core_row.index[0].split("_")[0].split("-")[1] == root_core: #if the second part of the pair is the root core, return the first part
            root_core_pair = root_core_row.index[0].split("_")[0].split("-")[0]
    else: #just in case
        sys.exit(f"ERROR: Could not determine root core pair for root core {root_core}.")

    return root_core_pair

def find_matches(index, core_name): # Find the matches for a given index, core_name. Outputs a list of matching indices and their associated core names.
    match_indices = []  # empty array
    match_cores = [] # empty array

    row_vals = big_table.loc[index] #get the row values for the given index
    non_zero_columns = row_vals[(row_vals.notna()) & (row_vals != 0)].index.tolist() #get the non-zero columns for the given index

    #pick the active column for this core from the non-zero columns, since only 2 active columns per row
    core_columns = [col for col in non_zero_columns if col.endswith(core_name)] #find the column that ends with the core name, which is the core we are looking for matches for
    if not core_columns: #just in case
        print(f"No active columns found for core {core_name} in row {index}.")
        sys.exit()

    column_name = core_columns[0]
    core_value = big_table.at[index, column_name]  # get value of core in current row

    #below filters 
    for column in big_table.columns:  # for every column in big_table

        if column.endswith(core_name): #make sure column is same core
            for idx, value in big_table[column].items(): #for every value in the column
                if pd.notna(value) and abs(value - core_value) <= error_margin: #if value is not NA and within error margin, add index to match_indices
                    match_indices.append(idx)

                    col_check1 = column.split("-")[0]
                    col_check2 = column.split("-")[1].split("_")[0]
                    if col_check1 != core_name: #if the first part of the pair is not the core name, then it must be the matching core
                        matching_core = col_check1
                    elif col_check2 != core_name: #if the second part of the pair is not the core name, then it must be the matching core
                        matching_core = col_check2
                    else:
                        print(f"ERROR: Could not determine matching core for column {column} in row {index}.")
                        sys.exit()
                    match_cores.append(matching_core)
                    
    return match_indices, match_cores

@lru_cache(maxsize=None)
def find_matches_cached(index, core_name): #cache the find_matches function to speed up calls
    return find_matches(index, core_name)

def walk_back(current_floor_number, network): #walk back to root to get path. 
    if current_floor_number == 0: #no path on first floor, since it is the root.
        return [], []
    path = [] #blank indices and path
    
    current_slot = len([n for n in network if n[0] == current_floor_number]) #get current slot in network for this floor, which is just the total number of nodes.

    for floor in range(current_floor_number-1, -1, -1): #walk back through floors
        addition_mod = 0 #the addition modifier
        floor_nodes = [n for n in network if n[0] == floor] # get the nodes for this floor
        for i, fnode in enumerate(floor_nodes): #for each node in the floor
            addition_mod += fnode[3] #add the number of matches for this node to the addition modifier
            if addition_mod >= current_slot+1: # if the addition modifier is greater than or equal to the current slot, we have found the parent node
                path.append(fnode) #add the parent node to the path
                current_slot = i #update the current slot to the index of the parent node 
                break #break to move to next floor

    path = [(p[1], p[2]) for p in path] #convert path to list of tuples of (index, core) for readability
    indices = [idx for idx, _ in path] #return the indices from the existing path
    return indices, path

def bfs(root_index, root_core, *, max_floor=20): #breadth-first-search
    root = (root_index, root_core) #create root node as a tuple of (index, core)
    root_pair = (root_index, get_core_pair(root)) #create root pair as a tuple of (index, core pair) using the get_core_pair function to find the core pair for the root core

    queue = deque([root_pair]) #this uses deque, which is a double-ended queue that allows for efficient appending and popping from both ends, to create the initial queue with the root pair as the only element.
    next_floor = deque() #this will be used to store the nodes for the next floor as we find matches for the current floor
    network = [] #creates empty network variable
    current_floor_number = 0 #start at floor zero, which is the root node
    export_pathways = [] #store the pathways found for export

    while queue: #while there are still nodes to explore in the queue

        if current_floor_number > max_floor: #if we have exceeded the maximum floor,
            break

        floor_width = len(queue) #the width of the current floor is the number of nodes in the queue at the start of the floor
        for _ in range(floor_width): #for each node in the current floor
            node = queue.popleft() #current node removes from left of queue
            idx, core = node #unpack the node into index and core

            extracted_numbers, existing_path = walk_back(current_floor_number, network) #walk back and get all the indices and the full path

            if idx in extracted_numbers: #if the index of the current node is already in the existing path, we have looped back around to a previously visited node, so we can stop exploring this path and add it to the export pathways
                match_indices, match_cores = [], [] #clear match indices and cores since we are not exploring this path anymore
                existing_path.insert(0, node) #insert the node at beginning to preserve order
                existing_path.reverse() #reverse to get correct order from root to leaf
                export_pathways.append((root, existing_path)) #add the root and the existing path to the export pathways
            #DEBUG TESTING BELOW #TODO
            elif core in (c for i, c in existing_path): #if core is in the existing path, then we have visited a copy before, and need to verify if we end or not
                visited_same_core = next((node for node in existing_path if node[1] == core), None) #get the core that we have previously visited
                same_core_depth = get_depth(visited_same_core[1], visited_same_core[0])
                current_core_depth = get_depth(core, idx)
                if abs(same_core_depth-current_core_depth) < error_margin:
                    match_indices, match_cores = [], [] #clear match indices and cores since we are not exploring this path anymore
                    existing_path.insert(0, node) #insert at beginning to preserve order
                    existing_path.reverse() #reverse to get correct order from root to leaf
                    export_pathways.append((root, existing_path)) #add the root and the existing path to the export pathways
            #DEBUG TESTING ABOVE #TODO
            elif core == root_core: #if the core is the same as the root core, we have found a loop back to the root, so we can add this path to the export pathways
                match_indices, match_cores = [], [] #clear match indices and cores since we are not exploring this path anymore
                existing_path.insert(0, node) #insert at beginning to preserve order
                existing_path.reverse() #reverse to get correct order from root to leaf
                export_pathways.append((root, existing_path)) #add the root and the existing path to the export pathways
            elif core != root_core or current_floor_number == 0: #if the core is not the root core, or if we are on the first floor (since the first floor is the root node and we want to find matches for it), find matches and add them to the next floor
                match_indices, match_cores = find_matches_cached(idx, core) #find matches for the current node
                next_floor.extend(zip(match_indices, match_cores)) #add matches to next floor

            network.append((current_floor_number, idx, core, len(match_indices)))  #the full network, saved but not used.
        
        current_floor_number += 1 #increment floor number after exploring all nodes in the current floor
        queue.extend(next_floor) #add the nodes for the next floor to the queue to explore in the next iteration of the while loop
        next_floor.clear() #clear the next floor for the next iteration

    
    return export_pathways, network

In [ ]:
sum = 0
all_naughty = set()

for code_index, code_row in tqdm(code_networks.iterrows(), total=len(code_networks), desc="Processing BFS", unit="row", position=0): #for each row in the code_networks dataframe, which contains the codes for the networks with errors
    code_row = code_row.dropna() #drop NA, so codes only.
    if code_index not in error_indices: #skip non-error rows
        continue
    
    network_codes = code_row.tolist() #put codes in list

    code_cols = original_big_table.filter(regex=r'_code$').columns #all columns in original_big_table that end with _code
    big_table = original_big_table[original_big_table[code_cols].isin(network_codes).any(axis=1)] #filter to only include rows where any of the code columns match the network codes for this row
    find_matches_cached.cache_clear() # cache depends on global big_table; clear per-network to avoid stale indices

    #--- BFS search below.

    every_path = []
    for index, row in big_table.iterrows(): #for each row in big_table
        row = row.dropna() #drop NA values to avoid confusion
        core1 = row.index[0].split("-")[0] #names of the cores
        core2 = row.index[0].split("-")[1].split("_")[0]

        for core in (core1, core2): #for each core

            paths, network = bfs(index, core, max_floor=6) #get paths and network from BFS
            for p in paths:
                every_path.append(p)

    #--- Depths/naughty confirmation below.

    error_networks = 0 #number of error networks
    circular_networks = 0 #total number of circular networks

    naughty_list = set() #use sets for unique values
    nice_list = set()

    for i, path in enumerate(every_path): #for each path in every_path, get the root core and end node core, and if they are the same, check if the depth difference is within the error margin.
        
        root_core = every_path[i][0][1] #gets the root core
        root_index = every_path[i][0][0] #gets the root index

        end_node_core = every_path[i][1][-1][1] #gets the end node core, which is the core of the last node in the path
        end_node_index = every_path[i][1][-1][0] #get the end node index, which is the index of the last node in the path

        all_indices = [node[0] for node in every_path[i][1]] #gets all the indices in the path, which will be used to add to the naughty or nice list depending on whether this path is an error network or not

        if root_core != end_node_core: #if not circular, skip
            continue

        if len(all_indices) < 3: #if the path is too short to be successful, skip
            continue

        else:
            circular_networks += 1 #add to toal cicular
            root_depth = get_depth(root_core, root_index) #get the root depth
            end_node_depth = get_depth(end_node_core, end_node_index) #get the end node depth
            if abs(root_depth - end_node_depth) >= error_margin: #if depths differ more than by error margin, error network
                error_networks += 1

                for idx in all_indices: #put all the indices in the naughty list, unless in nice list
                    if idx in nice_list:
                        continue
                    else:
                        naughty_list.add(idx)

            elif abs(root_depth - end_node_depth) <= error_margin: #else, everything into nice list
                for idx in all_indices:
                    nice_list.add(idx)
                    if idx in naughty_list:
                        naughty_list.remove(idx)
    if len(naughty_list) == 1: #if there is only one naughty
        all_naughty.add((get_code(naughty_list.pop()), code_index))
    # if code_index == 618:
    #     print(f"naughty: {naughty_list}, code: {get_code(naughty_list.pop())}")
    #     print(f"naughty: {naughty_list}")
    #     print(f'Index {code_index}: Naughty List Length: {len(naughty_list)}, Nice List Length: {len(nice_list)}, Error Networks: {error_networks}, Circular Networks: {circular_networks}')
print(len(all_naughty))

Processing BFS:  40%|████      | 1027/2562 [00:03<00:04, 310.18row/s]

2893.14
2893.14
2893.14
2893.14
2893.14
2893.14
2893.14
2893.14
2893.14
2893.14
1222.1812
2893.319
2893.319
2893.319
2893.319
2893.319
2893.3295
2893.3295
2893.3295
2893.3295
2893.3295


Processing BFS: 100%|██████████| 2562/2562 [00:05<00:00, 480.86row/s]

32


In [130]:
sorted_all_naughty = sorted(all_naughty, key=lambda x: x[1])
sorted_all_naughty

[('WDC-TALDICE_162', 327),
 ('WDC-EDML_263', 437),
 ('WDC-TALDICE_216', 450),
 ('WDC-TALDICE_236', 465),
 ('WDC-TALDICE_240', 476),
 ('WDC-TALDICE_258', 501),
 ('WDC-TALDICE_279', 586),
 ('EDC-EDML_105', 604),
 ('WDC-EDML_375', 605),
 ('WDC-TALDICE_312', 643),
 ('WDC-TALDICE_316', 657),
 ('WDC-DF_263', 684),
 ('WDC-TALDICE_330', 749),
 ('WDC-TALDICE_332', 762),
 ('EDC-WDC_365', 763),
 ('WDC-EDML_481', 779),
 ('WDC-EDML_480', 780),
 ('WDC-DF_313', 805),
 ('EDC-EDML_156', 848),
 ('EDC-WDC_452', 949),
 ('WDC-EDML_700', 1057),
 ('WDC-DF_438', 1088),
 ('WDC-TALDICE_508', 1129),
 ('WDC-TALDICE_520', 1143),
 ('WDC-TALDICE_528', 1157),
 ('EDC-DF_242', 1166),
 ('WDC-TALDICE_544', 1186),
 ('EDC-DF_276', 1256),
 ('EDC-TALDICE_180', 1304),
 ('WDC-TALDICE_623', 1402),
 ('WDC-DF_569', 1406),
 ('WDC-EDML_926', 1416)]

In [131]:
reference, index, depth1, depth2 = get_tie_info('WDC-DF_569', depths=True)

fully_wrong = []
for code, code_index in sorted_all_naughty:
    reference, index, depth1, depth2 = get_tie_info(code, depths=True)
    fully_wrong.append(index)

pd.Series(fully_wrong).to_csv('/Users/quinnmackay/Desktop/wrong.txt', index=False, header=None)

WDC-DF_WDC             3339.11
WDC-DF_DF               983.54
WDC-DF_reference    DF21_Chron
WDC-DF_code         WDC-DF_569
Name: 4287, dtype: object
WDC-TALDICE_WDC                             1836.6951
WDC-TALDICE_TALDICE                          619.3798
WDC-TALDICE_reference    ChristoWesterlies; SiglEmail
WDC-TALDICE_code                      WDC-TALDICE_162
Name: 4467, dtype: object
WDC-EDML_WDC                  2086.1328
WDC-EDML_EDML                  732.5393
WDC-EDML_reference    ChristoWesterlies
WDC-EDML_code              WDC-EDML_263
Name: 3007, dtype: object
WDC-TALDICE_WDC                2133.6763
WDC-TALDICE_TALDICE            725.68353
WDC-TALDICE_reference          SiglEmail
WDC-TALDICE_code         WDC-TALDICE_216
Name: 4521, dtype: object
WDC-TALDICE_WDC                 2190.918
WDC-TALDICE_TALDICE              736.122
WDC-TALDICE_reference          SiglEmail
WDC-TALDICE_code         WDC-TALDICE_236
Name: 4541, dtype: object
WDC-TALDICE_WDC                2219.8335
W

In [132]:
original_big_table

,EDC-WDC_EDC,EDC-WDC_WDC,EDC-WDC_reference,EDC-WDC_code,EDC-EDML_EDC,EDC-EDML_EDML,EDC-EDML_reference,EDC-EDML_code,EDC-DF_EDC,EDC-DF_DF,...,EDML-DF_reference,EDML-DF_code,EDML-TALDICE_EDML,EDML-TALDICE_TALDICE,EDML-TALDICE_reference,EDML-TALDICE_code,DF-TALDICE_DF,DF-TALDICE_TALDICE,DF-TALDICE_reference,DF-TALDICE_code
0,21.1011,100.4808,ChristoWesterlies; SiglEmail,EDC-WDC_0,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,22.0379,105.2728,ChristoWesterlies; SiglEmail,EDC-WDC_1,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,23.1031,110.2635,ChristoWesterlies; SiglEmail,EDC-WDC_2,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,29.6430,142.9708,ChristoWesterlies; SiglEmail,EDC-WDC_3,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,29.8606,144.1818,ChristoWesterlies; SiglEmail,EDC-WDC_4,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5396,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,1002.18,1252.1900,Svensson_Links,DF-TALDICE_106
5397,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,1010.42,1256.5894,Svensson_Links,DF-TALDICE_107
5398,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,1012.31,1257.3143,Svensson_Links,DF-TALDICE_108
5399,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,1016.07,1259.1330,Svensson_Links,DF-TALDICE_109
